# Диагностика маппинга ССП (CDI/OCRM)

Цель тетрадки:
- проверить качество источника `ocrm_ul.s_org_ext`;
- понять, можно ли тянуть `ССП` из `x_area_resp` по `x_inn`;
- показать, как это влияет на покрытие договоров.

Запуск: выполняйте ячейки сверху вниз и присылайте скриншоты результатов.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect
from connection_secrets import LAKE_USER, LAKE_PASSWORD

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={
        'user_name': LAKE_USER,
        'password': LAKE_PASSWORD,
    }
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
# Параметры проверки
target_inn = '401400143905'
target_cdi_id = '604399'   # если не нужен, оставьте как есть
month_start = '2026-02-01' # для покрытия договорного контура
month_end = '2026-02-29'   # не обязательно первое число

print('target_inn   =', target_inn)
print('target_cdi_id=', target_cdi_id)
print('month range  =', month_start, '->', month_end)

## 1) Проверка структуры и доступности источников

In [ ]:
with imp:
    s_org_ext_desc = imp.fetch('describe ocrm_ul.s_org_ext')
    s_org_ext_cnt = imp.fetch('select count(*) as cnt from ocrm_ul.s_org_ext')

display(s_org_ext_desc)
display(s_org_ext_cnt)

In [ ]:
# Проверка таблицы bridge (если существует в вашем контуре)
with imp:
    ext_id_org_desc = imp.fetch('describe cdiul.ext_id_org')
    ext_id_org_cnt = imp.fetch('select count(*) as cnt from cdiul.ext_id_org')

display(ext_id_org_desc)
display(ext_id_org_cnt)

## 2) Диагностика дублей и конфликтов в `ocrm_ul.s_org_ext`

In [ ]:
# 2.1 Сколько INN имеют дубли (rows > 1)
with imp:
    dup_inn_top = imp.fetch("""
        select
            cast(x_inn as string) as x_inn,
            count(*) as cnt
        from ocrm_ul.s_org_ext
        where x_inn is not null
        group by cast(x_inn as string)
        having count(*) > 1
        order by cnt desc
        limit 100
    """)

display(dup_inn_top)

In [ ]:
# 2.2 Для дублей: одинаковый ли ССП, или есть конфликт значений x_area_resp
with imp:
    dup_inn_ssp_quality = imp.fetch("""
        with inn_stats as (
            select
                cast(x_inn as string) as x_inn,
                count(*) as rows_cnt,
                count(distinct cast(x_area_resp as string)) as ssp_variants_cnt
            from ocrm_ul.s_org_ext
            where x_inn is not null
            group by cast(x_inn as string)
            having count(*) > 1
        )
        select
            count(*) as duplicated_inn_cnt,
            sum(case when ssp_variants_cnt = 1 then 1 else 0 end) as tech_dups_same_ssp,
            sum(case when ssp_variants_cnt > 1 then 1 else 0 end) as conflicting_ssp_inn
        from inn_stats
    """)

display(dup_inn_ssp_quality)

In [ ]:
# 2.3 Детально по вашему INN
with imp:
    target_inn_rows = imp.fetch(f"""
        select
            cast(x_inn as string) as x_inn,
            cast(x_cdi_id as string) as x_cdi_id,
            cast(x_area_resp as string) as x_area_resp,
            *
        from ocrm_ul.s_org_ext
        where cast(x_inn as string) = '{target_inn}'
    """)

display(target_inn_rows)
print('rows_found =', len(target_inn_rows))

## 3) Проверка bridge `cdiul.ext_id_org` (CDI <-> OCRM)

Если эта связка работает, это лучший технический путь маппинга.

In [ ]:
# 3.1 Что есть по вашему CDI в s_org_ext
with imp:
    target_cdi_rows = imp.fetch(f"""
        select
            cast(x_inn as string) as x_inn,
            cast(x_cdi_id as string) as x_cdi_id,
            cast(x_area_resp as string) as x_area_resp
        from ocrm_ul.s_org_ext
        where cast(x_cdi_id as string) = '{target_cdi_id}'
    """)

display(target_cdi_rows)
print('rows_found =', len(target_cdi_rows))

In [ ]:
# 3.2 Проверка наличия OCRM-маппинга в bridge
with imp:
    bridge_sample = imp.fetch("""
        select
            cmo_ext_source_system,
            count(*) as cnt
        from cdiul.ext_id_org
        group by cmo_ext_source_system
        order by cnt desc
        limit 50
    """)

display(bridge_sample)

## 4) Покрытие договорного контура ССП по INN

Без витрины: берем договоры из `agreements + companies`, затем тянем `x_area_resp` из OCRM по INN.

In [ ]:
with imp:
    coverage_by_inn = imp.fetch(f"""
        with agr_scope as (
            select
                cast(a.abs_agr_id as string) as agr_id,
                cast(c.c_inn as string) as inn,
                a.c_agr_number,
                a.d_valid_from,
                a.d_valid_to
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and coalesce(cast(a.d_valid_to as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
        ),
        ocrm_by_inn as (
            select
                cast(x_inn as string) as inn,
                min(cast(x_area_resp as string)) as ssp_any,
                count(distinct cast(x_area_resp as string)) as ssp_variants_cnt
            from ocrm_ul.s_org_ext
            where x_inn is not null
              and x_area_resp is not null
              and trim(cast(x_area_resp as string)) <> ''
            group by cast(x_inn as string)
        ),
        final_map as (
            select
                a.agr_id,
                a.inn,
                a.c_agr_number,
                case when o.ssp_variants_cnt = 1 then o.ssp_any else null end as ssp,
                o.ssp_variants_cnt
            from agr_scope a
            left join ocrm_by_inn o
              on o.inn = a.inn
        )
        select
            count(*) as agr_total,
            sum(case when ssp is not null then 1 else 0 end) as ssp_filled_cnt,
            sum(case when ssp is null then 1 else 0 end) as ssp_empty_or_conflict_cnt,
            sum(case when ssp_variants_cnt > 1 then 1 else 0 end) as ssp_conflict_cnt,
            round(100.0 * sum(case when ssp is not null then 1 else 0 end) / count(*), 2) as ssp_fill_rate_pct
        from final_map
    """)

display(coverage_by_inn)

In [ ]:
# Детально по целевому INN внутри договорного контура
with imp:
    target_inn_contracts = imp.fetch(f"""
        with agr_scope as (
            select
                cast(a.abs_agr_id as string) as agr_id,
                cast(c.c_inn as string) as inn,
                a.c_agr_number,
                a.d_valid_from,
                a.d_valid_to
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and cast(c.c_inn as string) = '{target_inn}'
        ),
        ocrm_by_inn as (
            select
                cast(x_inn as string) as inn,
                cast(x_area_resp as string) as ssp
            from ocrm_ul.s_org_ext
            where cast(x_inn as string) = '{target_inn}'
        )
        select
            a.*,
            o.ssp
        from agr_scope a
        left join ocrm_by_inn o
          on o.inn = a.inn
        order by a.c_agr_number
    """)

display(target_inn_contracts)

## 5) Интерпретация

- Если `ssp_conflict_cnt = 0`, маппинг по INN можно считать рабочим.
- Если `ssp_conflict_cnt > 0`, нужно правило актуальности (дата/active) для выбора одной записи.
- Если по вашему `target_inn` несколько `x_area_resp`, присылайте скрин результата ячеек 2.3 и последней — подберем точное правило дедупликации.